# Day 42: Datasets, Features, Labels and Data Quality

**Today's question:** a model learns from data. Before we let it learn, **can we trust this file?**

Yesterday we used `students.csv`, which was already clean. Today we open the raw file it came from,
`students_raw.csv`, and audit it with plain Python. Before running a cell marked 🤔, predict what it will show.

## 1. Rows and columns

In [ ]:
import csv
from pathlib import Path

DATA_DIR = Path("../../../datasets")

with open(DATA_DIR / "students_raw.csv", encoding="utf-8", newline="") as f:
    rows = list(csv.DictReader(f))

columns = list(rows[0].keys())
print(len(rows), "rows,", len(columns), "columns")
print(columns)

In [ ]:
rows[0]

| Word | Means | Here |
|------|-------|------|
| **row** (example, record) | one thing we observed | one student |
| **column** (attribute) | one property measured for every row | `study_hours` |
| **feature** | a column the model may use as *input* | `study_hours`, `program`, … |
| **target** (label) | the column we want to *predict* | `final_score` or `passed` |

## 2. Features vs target

| Column | Role | Why |
|--------|------|-----|
| `student_id` | **identifier**: not a feature | just a name tag; a model could memorise it instead of learning |
| `program` … `has_job` | **features** | facts known *before* the exam |
| `final_score` | **target** for predicting a number (regression) | what we want to predict |
| `passed` | **target** for predicting a category (classification) | 1 if `final_score >= 50` |

🤔 **Discuss:** if we want to predict `passed`, may we use `final_score` as a feature? (Hint: how is `passed` made?)

## 3. Numeric vs categorical

- **Numeric:** a quantity; averages make sense (`study_hours`, `attendance_pct`).
- **Categorical:** one of a few labels; averages make no sense (`program`, `has_job`).

Counting the **distinct values** in each column is a quick first clue.

In [ ]:
for column in columns:
    distinct = set([row[column] for row in rows])
    print(f"{column:<17} {len(distinct):>4} distinct values")

🤔 **Predict:** `program` should have 3 values (science, commerce, arts) and `has_job` 2 (yes, no).
Why does the table show 10 and 8? Keep that question for section 5.

Careful: `passed` is stored as `0`/`1` but it is **categorical** (fail/pass). Numbers are not always quantities.

## 4. Missing values

A missing value in a CSV file is an **empty string** `""`, not `None`.

In [ ]:
# Try to compute the average study time. Why does it crash?
total = 0
for row in rows:
    total += float(row["study_hours"])

`float("")` cannot work. Let's count the blanks in every column instead.

In [ ]:
missing = {}
for column in columns:
    count = 0
    for row in rows:
        if row[column].strip() == "":
            count += 1
    missing[column] = count

print(missing)

## 5. Inconsistent categories

🤔 **Predict:** how many different spellings of the program names will we find?

In [ ]:
program_counts = {}
for row in rows:
    value = row["program"]
    program_counts[value] = program_counts.get(value, 0) + 1

for value, count in program_counts.items():
    print(f"'{value}': {count}")   # quotes make hidden spaces visible

To a computer, `'Arts'`, `'ARTS'` and `'arts '` are three different categories. A model would treat them as three
different programs. Normalising (strip spaces + lower case) shows what was meant:

In [ ]:
clean_counts = {}
for row in rows:
    value = row["program"].strip().lower()
    clean_counts[value] = clean_counts.get(value, 0) + 1

print(clean_counts)

## 6. Duplicate rows

An **exact duplicate** has the same value in every column. A tuple of the row's values can go into a set
(dicts cannot), so we can remember which rows we have already seen.

In [ ]:
seen = set()
duplicate_ids = []
for row in rows:
    key = tuple(row.values())
    if key in seen:
        duplicate_ids.append(row["student_id"])
    seen.add(key)

print(len(duplicate_ids), "exact duplicates:", duplicate_ids)

🤔 **Discuss:** why are duplicates a problem for a model? (Hint: what if one student appears 50 times?)

## 7. Out-of-range values

Some values are *present* but **impossible**: attendance above 100 %, negative hours. We need:

1. a safe converter that turns `""` into `None` instead of crashing (Day 25 `try/except`);
2. the allowed range for each numeric column (we decide these using common sense).

In [ ]:
def to_number(text):
    """Return text as a float, or None if it is blank or not a number."""
    try:
        return float(text)
    except ValueError:
        return None


print(to_number("7.5"), to_number(""), to_number("seven"))

In [ ]:
allowed = {
    "study_hours": (0, 80),
    "attendance_pct": (0, 100),
    "sleep_hours": (0, 24),
    "previous_score": (0, 100),
    "assignments_done": (0, 10),
    "final_score": (0, 100),
}

for column in allowed:
    low, high = allowed[column]
    for row in rows:
        value = to_number(row[column])
        if value is not None and (value < low or value > high):
            print(row["student_id"], column, value)

`value is not None` skips the blanks: missing is a *different* problem from out of range, and we count it separately.

## 8. The data-quality audit table

Put it all together: one line per column, plus the duplicates.

In [ ]:
def count_out_of_range(rows, column, low, high):
    """Count values in a column that are present but outside low..high."""
    count = 0
    for row in rows:
        value = to_number(row[column])
        if value is not None and (value < low or value > high):
            count += 1
    return count


print(f"{'column':<17} {'missing':>7} {'out of range':>12} {'distinct':>8}")
for column in columns:
    distinct = len(set([row[column] for row in rows]))
    if column in allowed:
        low, high = allowed[column]
        bad = count_out_of_range(rows, column, low, high)
    else:
        bad = "-"
    print(f"{column:<17} {missing[column]:>7} {bad:>12} {distinct:>8}")
print(f"\nExact duplicate rows: {len(duplicate_ids)}")

### What should we do about each problem? (We decide on Day 45, not today.)

| Problem | Found | Possible fixes |
|---------|-------|----------------|
| missing `study_hours` / `sleep_hours` | 8 / 6 | drop those rows, or **fill** with a typical value |
| inconsistent `program` / `has_job` | 10 / 8 spellings | strip spaces + lower case |
| exact duplicates | 4 | keep one copy |
| attendance 150 %, study −3 h | 1 + 1 | ask the source; if impossible, remove the row |

**Golden rule:** never edit the raw file by hand. Write cleaning *code*, so every step is visible and repeatable.

## Recap

1. Rows are examples; columns are features or the target (label). IDs are not features.
2. Numeric = quantities; categorical = labels (even when stored as 0/1).
3. Audit before you analyse: missing values, inconsistent categories, duplicates, out-of-range values.
4. In a CSV, "missing" means an empty string. Convert safely with `try/except`.

**Next time:** NumPy. Doing maths on a whole column at once, without loops.